In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251118_011127"


In [2]:
# 1. Setup de caminhos locais
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
RAW_PATH: C:\Users\ander\OneDrive\TCC_USP\data_raw


In [3]:
# 2. Parametros globais e utilitarios
import pandas as pd
import yfinance as yf
from datetime import datetime

PERIODO = cfg.get_periodo_estudo()
START_DATE = PERIODO["start"]
END_DATE = PERIODO["end"]
TICKERS = {"ibovespa": "^BVSP", "bova11": "BOVA11.SA"}

os.makedirs(RAW_PATH, exist_ok=True)
print(f"Periodo configurado: {START_DATE} -> {END_DATE}")
print("Tickers alvo:", TICKERS)


Periodo configurado: 2018-01-01 -> 2025-01-31
Tickers alvo: {'ibovespa': '^BVSP', 'bova11': 'BOVA11.SA'}


In [4]:
# 3. Funcoes para download via yfinance
from typing import Tuple

def download_ticker(name: str, ticker: str, start: str, end: str) -> Tuple[Path, pd.DataFrame]:
    print(f"Baixando {ticker} ({start} -> {end})...")
    data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if data.empty:
        raise ValueError(f"Sem dados retornados para {ticker} no periodo especificado.")
    data = data.reset_index().rename(columns={"Date": "day", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Adj Close": "adj_close", "Volume": "volume"})
    data["source_ticker"] = ticker
    target_path = Path(RAW_PATH) / f"{name}.csv"
    data.to_csv(target_path, index=False)
    print(f"Salvo em {target_path}")
    return target_path, data


In [5]:
# 4. Executar downloads para todos os tickers
summary_rows = []
for name, ticker in TICKERS.items():
    path, df = download_ticker(name, ticker, START_DATE, END_DATE)
    summary_rows.append({"arquivo": path.name, "ticker": ticker, "records": len(df), "inicio": df["day"].min(), "fim": df["day"].max()})
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


Baixando ^BVSP (2018-01-01 -> 2025-01-31)...


Salvo em C:\Users\ander\OneDrive\TCC_USP\data_raw\ibovespa.csv
Baixando BOVA11.SA (2018-01-01 -> 2025-01-31)...


Salvo em C:\Users\ander\OneDrive\TCC_USP\data_raw\bova11.csv


,arquivo,ticker,records,inicio,fim
0,ibovespa.csv,^BVSP,1758,2018-01-02,2025-01-30
1,bova11.csv,BOVA11.SA,1741,2018-01-02,2025-01-30


In [6]:
# 5. Listar arquivos gerados em data_raw
for path in Path(RAW_PATH).glob('*.csv'):
    size_kb = path.stat().st_size / 1024
    print(f" - {path.name}: {size_kb:0.1f} KB")


 - bova11.csv: 199.2 KB
 - ibovespa.csv: 120.0 KB
 - noticias_exemplo.csv: 0.8 KB
 - noticias_real.csv: 91.7 KB
 - noticias_real_dummy.csv: 0.4 KB
